# 04 · ETL vs ELT
- **ETL**: Extract → *Transform in the pipeline (pandas/Spark)* → Load the finished result.
- **ELT**: Extract → *Load the raw data first* → Transform inside the warehouse with SQL.

Same answer, different place where the work happens. ELT keeps raw data around (re-transform any time) and lets the warehouse engine do the heavy lifting.

In [ ]:
import sys
sys.path.insert(0, "..")          # lets the notebook import common.py from python_practice/
from common import *              # load_operations, check, OUT, ...
import pandas as pd

df = load_operations()            # reads the dashboard API if it is running, else data_samples/operations_clean.csv
import duckdb

## ETL: transform in pandas

In [ ]:
etl = (df.groupby("category").agg(revenue=("revenue", "sum"), profit=("profit", "sum"), records=("fact_id", "count"))
         .assign(margin_pct=lambda t: (t["profit"] / t["revenue"] * 100).round(2)).round(2).sort_index())
etl

## ELT: load raw, transform with SQL in DuckDB
DuckDB plays the warehouse here. The raw table stays untouched; the transformation is just a query (or a view).

In [ ]:
wh = duckdb.connect()                      # in-memory warehouse
wh.register("raw_ops", df)                 # "load" step: raw data lands as-is
elt = wh.sql("""
    SELECT category,
           ROUND(SUM(revenue), 2) AS revenue,
           ROUND(SUM(profit), 2)  AS profit,
           COUNT(*)               AS records,
           ROUND(SUM(profit) / SUM(revenue) * 100, 2) AS margin_pct
    FROM raw_ops GROUP BY category ORDER BY category
""").df().set_index("category")
elt

## Your turn
Write SQL against `raw_ops` that returns **total revenue per day_name**, with columns `day_name` and `revenue`, and store the DataFrame in `by_day`.

In [ ]:
by_day = None   # e.g. by_day = wh.sql("SELECT ...").df()

In [ ]:
same = etl["revenue"].round(2).equals(elt["revenue"].round(2))
check("ETL and ELT agree", same)
if by_day is not None:
    truth = df.groupby("day_name")["revenue"].sum().round(2)
    got = by_day.set_index("day_name")["revenue"].round(2)
    check("revenue per weekday", got.sort_index().equals(truth.sort_index()), "SELECT day_name, SUM(revenue) AS revenue FROM raw_ops GROUP BY day_name")